<h1>Bibliotecas</h1>
<i> - Coletar dados de fontes oficiais ligadas ao governo

In [2]:
import os
import time
import requests
import pandas as pd

from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path

<h1>Função padrão de exportação RAW</h1>

In [3]:
def salvar_raw(df_exportar, nome_base):
    nome_pipeline = "pipeline_verdadeiro_fontes_oficiais"

    data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    pasta_raw = Path(f"../dados/{nome_pipeline}/raw")
    pasta_raw.mkdir(parents=True, exist_ok=True)

    caminho_saida = pasta_raw / f"{nome_base}_raw_{data_agora}.csv"

    df_exportar.to_csv(
        caminho_saida,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Arquivo bruto salvo em: {caminho_saida}")
    
    print(f"Total de registros extraídos: {len(df_exportar)}")
    print(f"Data e hora da extração: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

<h1>Coleta de Dados - Sites</h1>

<h2>Camara dos Deputados</h2>

In [4]:
URL_CAMARA = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"

params_camara = {
    "ano": 2026,
    "itens": 20,
    "ordem": "DESC",
    "ordenarPor": "id"
}

resposta_camara = requests.get(URL_CAMARA, params=params_camara)

print(resposta_camara.status_code)

dados_camara = resposta_camara.json()

proposicoes = dados_camara.get("dados", [])

df_camara_raw = pd.DataFrame(proposicoes)

df_camara_raw.head()

salvar_raw(df_camara_raw, "camara_proposicoes")

200
Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\camara_proposicoes_raw_2026-05-10_02-18-24.csv
Total de registros extraídos: 20
Data e hora da extração: 10/05/2026 02:18:24


<h2>Senado</h2>

In [13]:
URL_SENADO = "https://legis.senado.leg.br/dadosabertos/materia/pesquisa/lista.json"

params_senado = {
    "ano": 2026
}

resposta_senado = requests.get(URL_SENADO, params=params_senado)

print(resposta_senado.status_code)
print(resposta_senado.url)

dados_senado = resposta_senado.json()

dados_senado.keys()

materias = dados_senado.get("PesquisaBasicaMateria", {}).get("Materias", {}).get("Materia", [])

df_senado_raw = pd.DataFrame(materias)

df_senado_raw.head()

df_senado_raw["fonte_verificacao"] = "SENADO_FEDERAL"
df_senado_raw["url_consulta"] = resposta_senado.url
df_senado_raw["data_coleta"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

salvar_raw(df_senado_raw, "senado_materias")

200
https://legis.senado.leg.br/dadosabertos/materia/pesquisa/lista.json?ano=2026
Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\senado_materias_raw_2026-05-10_02-28-13.csv
Total de registros extraídos: 1381
Data e hora da extração: 10/05/2026 02:28:13


<h2>TSE</h2>

In [ ]:
# =========================================================
# TSE - DADOS ABERTOS
# Dataset: Candidatos 2024
# Objetivo: coletar dados oficiais de candidaturas eleitorais
# =========================================================

import io
import zipfile

# =========================================================
# FUNÇÃO PADRÃO PARA SALVAR RAW
# Essa função salva qualquer DataFrame bruto na pasta correta
# =========================================================

def salvar_raw(df_exportar, nome_base):
    nome_pipeline = "pipeline_verdadeiro_fontes_oficiais"

    data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    pasta_raw = Path(f"../dados/{nome_pipeline}/raw")
    pasta_raw.mkdir(parents=True, exist_ok=True)

    caminho_saida = pasta_raw / f"{nome_base}_raw_{data_agora}.csv"

    df_exportar.to_csv(
        caminho_saida,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nArquivo bruto salvo em: {caminho_saida}")
    print(f"Total de registros extraídos: {len(df_exportar)}")
    print(f"Data e hora da extração: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")


# =========================================================
# STEP 1 — CONSULTAR O CATÁLOGO DO DATASET DO TSE
# Aqui a gente consulta o portal CKAN do TSE para descobrir
# quais arquivos existem dentro do dataset "candidatos-2024"
# =========================================================

URL_TSE = "https://dadosabertos.tse.jus.br/api/3/action/package_show"

params_tse = {
    "id": "candidatos-2024"
}

resposta_tse = requests.get(URL_TSE, params=params_tse)

print("Status da consulta:", resposta_tse.status_code)
print("URL consultada:", resposta_tse.url)
print("Tipo de conteúdo:", resposta_tse.headers.get("content-type"))


# =========================================================
# STEP 2 — CONVERTER A RESPOSTA PARA JSON
# Se a resposta não vier em JSON válido, o código para aqui
# =========================================================

try:
    dados_tse = resposta_tse.json()
    print("\nJSON carregado com sucesso.")
    print("Chaves principais:", dados_tse.keys())

except ValueError:
    print("\nA resposta do TSE não veio em JSON válido.")
    print("Prévia da resposta:")
    print(resposta_tse.text[:1000])
    raise


# =========================================================
# STEP 3 — EXTRAIR OS RECURSOS DO DATASET
# Cada recurso representa um arquivo disponível para download
# Exemplo: Candidatos, Bens de candidatos, Coligações etc.
# =========================================================

recursos_tse = dados_tse.get("result", {}).get("resources", [])

df_tse_recursos_raw = pd.DataFrame(recursos_tse)

print(f"\nTotal de recursos encontrados: {len(df_tse_recursos_raw)}")

display(df_tse_recursos_raw.head())


# =========================================================
# STEP 4 — ADICIONAR METADADOS AO CATÁLOGO DE RECURSOS
# Isso deixa o dado bruto rastreável
# =========================================================

df_tse_recursos_raw["fonte_verificacao"] = "TSE"
df_tse_recursos_raw["dataset_origem"] = "candidatos-2024"
df_tse_recursos_raw["url_consulta"] = resposta_tse.url
df_tse_recursos_raw["data_coleta"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")


# =========================================================
# STEP 5 — SALVAR O CATÁLOGO DOS RECURSOS COMO RAW
# Esse arquivo mostra quais recursos existiam no dataset
# no momento da coleta
# =========================================================

salvar_raw(df_tse_recursos_raw, "tse_candidatos_2024_recursos")


# =========================================================
# STEP 6 — FILTRAR SOMENTE RECURSOS EM FORMATO CSV
# O dataset do TSE pode ter JPEG, PDF, ZIP etc.
# Aqui vamos focar nos recursos tabulares
# =========================================================

df_tse_csv = df_tse_recursos_raw[
    df_tse_recursos_raw["format"].str.upper() == "CSV"
].copy()

print(f"\nTotal de recursos CSV encontrados: {len(df_tse_csv)}")

display(df_tse_csv[["name", "format", "description", "url"]].head(20))


# =========================================================
# STEP 7 — SELECIONAR O RECURSO PRINCIPAL: "CANDIDATOS"
# Esse é o arquivo oficial com dados de candidaturas
# =========================================================

recurso_candidatos = df_tse_csv[
    df_tse_csv["name"].str.strip().str.lower() == "candidatos"
]

if recurso_candidatos.empty:
    raise ValueError("Recurso 'Candidatos' não encontrado no catálogo do TSE.")

url_candidatos = recurso_candidatos.iloc[0]["url"]

print("\nURL do arquivo de candidatos:")
print(url_candidatos)


# =========================================================
# STEP 8 — BAIXAR O ARQUIVO REAL DO TSE
# Normalmente o TSE disponibiliza o CSV dentro de um ZIP
# =========================================================

resposta_arquivo_tse = requests.get(url_candidatos)

print("\nStatus do download:", resposta_arquivo_tse.status_code)
print("Tipo de conteúdo:", resposta_arquivo_tse.headers.get("content-type"))
print("Tamanho do arquivo em bytes:", len(resposta_arquivo_tse.content))

if resposta_arquivo_tse.status_code != 200:
    raise ValueError("Erro ao baixar o arquivo de candidatos do TSE.")


# =========================================================
# STEP 9 — ABRIR O ZIP EM MEMÓRIA
# Não precisamos salvar o ZIP manualmente no computador.
# O Python abre o conteúdo direto da resposta
# =========================================================

try:
    arquivo_zip = zipfile.ZipFile(io.BytesIO(resposta_arquivo_tse.content))
    arquivos_no_zip = arquivo_zip.namelist()

    print("\nArquivos encontrados dentro do ZIP:")
    print(arquivos_no_zip)

except zipfile.BadZipFile:
    raise ValueError("O arquivo baixado não parece ser um ZIP válido.")


# =========================================================
# STEP 10 — LOCALIZAR O CSV DENTRO DO ZIP
# Procuramos o primeiro arquivo com extensão .csv
# =========================================================

arquivos_csv = [
    nome for nome in arquivos_no_zip
    if nome.lower().endswith(".csv")
]

if not arquivos_csv:
    raise ValueError("Nenhum arquivo CSV encontrado dentro do ZIP.")

nome_csv = arquivos_csv[0]

print("\nCSV selecionado:")
print(nome_csv)


# =========================================================
# STEP 11 — LER O CSV DO TSE COM PANDAS
# O TSE geralmente usa separador ';' e encoding latin1
# =========================================================

with arquivo_zip.open(nome_csv) as arquivo:
    df_tse_candidatos_raw = pd.read_csv(
        arquivo,
        sep=";",
        encoding="latin1"
    )

print(f"\nTotal de candidatos/registros carregados: {len(df_tse_candidatos_raw)}")

display(df_tse_candidatos_raw.head())


# =========================================================
# STEP 12 — ADICIONAR METADADOS AO DATAFRAME DE CANDIDATOS
# Esses campos ajudam na rastreabilidade do dataset
# =========================================================

df_tse_candidatos_raw["fonte_verificacao"] = "TSE"
df_tse_candidatos_raw["dataset_origem"] = "candidatos-2024"
df_tse_candidatos_raw["url_arquivo"] = url_candidatos
df_tse_candidatos_raw["arquivo_origem"] = nome_csv
df_tse_candidatos_raw["data_coleta"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")


# =========================================================
# STEP 13 — SALVAR O RAW FINAL DE CANDIDATOS
# Esse é o arquivo bruto real com dados oficiais de candidaturas
# =========================================================

salvar_raw(df_tse_candidatos_raw, "tse_candidatos_2024")

Status da consulta: 200
URL consultada: https://dadosabertos.tse.jus.br/api/3/action/package_show?id=candidatos-2024
Tipo de conteúdo: application/json;charset=utf-8

JSON carregado com sucesso.
Chaves principais: dict_keys(['help', 'success', 'result'])

Total de recursos encontrados: 60


,cache_last_updated,cache_url,created,datastore_active,description,format,hash,id,last_modified,metadata_modified,mimetype,mimetype_inner,name,package_id,position,resource_type,size,state,url,url_type
0,None,None,2024-07-19T23:41:38.739905,False,Todas as UFs,CSV,,af76c401-0972-4ddf-8ea8-00e310ae53b4,None,2024-07-19T23:43:35.685561,application/zip,None,Candidatos,0bf7ed12-148e-41df-be3f-5d377e63635e,0,None,None,active,https://cdn.tse.jus.br/estatistica/sead/odsele...,None
1,None,None,2024-08-07T13:29:16.871078,False,Todas as UFs,CSV,,7a9e61af-2425-4f7a-acf2-19338e82d12c,None,2024-08-07T13:29:56.064902,application/zip,None,Candidatos - Informações complementares,0bf7ed12-148e-41df-be3f-5d377e63635e,1,None,None,active,https://cdn.tse.jus.br/estatistica/sead/odsele...,None
2,None,None,2024-07-19T23:43:35.694155,False,Todas as UFs,CSV,,2d078979-116f-498f-ac5b-2e2a6fb0ff1f,None,2024-07-19T23:44:27.201711,application/zip,None,Bens de candidatos,0bf7ed12-148e-41df-be3f-5d377e63635e,2,None,None,active,https://cdn.tse.jus.br/estatistica/sead/odsele...,None
3,None,None,2024-07-19T23:44:27.210483,False,Todas as UFs,CSV,,faa4bfd9-1ffd-413c-8154-1d2f71cd9a4c,None,2024-07-19T23:45:20.779049,application/zip,None,Coligações,0bf7ed12-148e-41df-be3f-5d377e63635e,3,None,None,active,https://cdn.tse.jus.br/estatistica/sead/odsele...,None
4,None,None,2024-07-19T23:45:20.788800,False,Todas as UFs,CSV,,7258e69e-68ac-4efc-9163-d083302407c8,None,2024-07-19T23:47:00.017466,application/zip,None,Vagas,0bf7ed12-148e-41df-be3f-5d377e63635e,4,None,None,active,https://cdn.tse.jus.br/estatistica/sead/odsele...,None



Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\tse_candidatos_2024_recursos_raw_2026-05-12_23-20-47.csv
Total de registros extraídos: 60
Data e hora da extração: 12/05/2026 23:20:47

Total de recursos CSV encontrados: 7


,name,format,description,url
0,Candidatos,CSV,Todas as UFs,https://cdn.tse.jus.br/estatistica/sead/odsele...
1,Candidatos - Informações complementares,CSV,Todas as UFs,https://cdn.tse.jus.br/estatistica/sead/odsele...
2,Bens de candidatos,CSV,Todas as UFs,https://cdn.tse.jus.br/estatistica/sead/odsele...
3,Coligações,CSV,Todas as UFs,https://cdn.tse.jus.br/estatistica/sead/odsele...
4,Vagas,CSV,Todas as UFs,https://cdn.tse.jus.br/estatistica/sead/odsele...
5,Motivo da Cassação,CSV,Todas as UFs,https://cdn.tse.jus.br/estatistica/sead/odsele...
6,Redes sociais de candidatos,CSV,Todas as UFs,https://cdn.tse.jus.br/estatistica/sead/odsele...



URL do arquivo de candidatos:
https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_2024.zip

Status do download: 200
Tipo de conteúdo: application/zip
Tamanho do arquivo em bytes: 63729440

Arquivos encontrados dentro do ZIP:
['leiame.pdf', 'consulta_cand_2024_AL.csv', 'consulta_cand_2024_SC.csv', 'consulta_cand_2024_TO.csv', 'consulta_cand_2024_SE.csv', 'consulta_cand_2024_RR.csv', 'consulta_cand_2024_RS.csv', 'consulta_cand_2024_SP.csv', 'consulta_cand_2024_RO.csv', 'consulta_cand_2024_PR.csv', 'consulta_cand_2024_RN.csv', 'consulta_cand_2024_PE.csv', 'consulta_cand_2024_BRASIL.csv', 'consulta_cand_2024_RJ.csv', 'consulta_cand_2024_PI.csv', 'consulta_cand_2024_PB.csv', 'consulta_cand_2024_MS.csv', 'consulta_cand_2024_MT.csv', 'consulta_cand_2024_PA.csv', 'consulta_cand_2024_MG.csv', 'consulta_cand_2024_ES.csv', 'consulta_cand_2024_MA.csv', 'consulta_cand_2024_GO.csv', 'consulta_cand_2024_CE.csv', 'consulta_cand_2024_BA.csv', 'consulta_cand_2024_AP.csv', 'consul

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO
0,09/05/2026,19:30:12,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,6,ENSINO MÉDIO COMPLETO,3,CASADO(A),3,PARDA,601,AGRICULTOR,4,NÃO ELEITO
1,09/05/2026,19:30:12,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,6,ENSINO MÉDIO COMPLETO,1,SOLTEIRO(A),2,PRETA,129,ARTESÃO,5,SUPLENTE
2,09/05/2026,19:30:12,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,6,ENSINO MÉDIO COMPLETO,3,CASADO(A),3,PARDA,601,AGRICULTOR,3,ELEITO POR MÉDIA
3,09/05/2026,19:30:12,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,6,ENSINO MÉDIO COMPLETO,1,SOLTEIRO(A),3,PARDA,601,AGRICULTOR,5,SUPLENTE
4,09/05/2026,19:30:12,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,4,ENSINO FUNDAMENTAL COMPLETO,3,CASADO(A),3,PARDA,999,OUTROS,5,SUPLENTE



Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\tse_candidatos_2024_raw_2026-05-12_23-21-02.csv
Total de registros extraídos: 5834
Data e hora da extração: 12/05/2026 23:21:02


<h2>Portal da Transparencia</h2>

<h3>Contratos públicos</h3>
<i> - rodar 1x por semana

In [17]:
# =========================================================
# PORTAL DA TRANSPARÊNCIA - CONTRATOS COM PAGINAÇÃO E ANOS
# Objetivo: coletar contratos públicos de vários anos
# Fonte: Portal da Transparência
# =========================================================


load_dotenv("../portal-transparencia-api-key.env")

API_KEY_TRANSPARENCIA = os.getenv("PORTAL_TRANSPARENCIA_API_KEY")

if not API_KEY_TRANSPARENCIA:
    raise ValueError("Chave do Portal da Transparência não encontrada.")

# =========================================================
# STEP 2 — CONFIGURAR ENDPOINT E HEADER
# =========================================================

URL_CONTRATOS = "https://api.portaldatransparencia.gov.br/api-de-dados/contratos"

headers_transparencia = {
    "chave-api-dados": API_KEY_TRANSPARENCIA
}

# =========================================================
# STEP 3 — DEFINIR ÓRGÃOS E ANOS DE CONSULTA
# =========================================================

orgaos_consulta = [
    {
        "codigo_orgao": "26000",
        "nome_orgao": "MINISTERIO_DA_EDUCACAO"
    },
    {
        "codigo_orgao": "36000",
        "nome_orgao": "MINISTERIO_DA_SAUDE"
    },
    {
        "codigo_orgao": "30000",
        "nome_orgao": "MINISTERIO_DA_JUSTICA"
    },
    {
        "codigo_orgao": "55000",
        "nome_orgao": "MINISTERIO_DO_DESENVOLVIMENTO_SOCIAL"
    }
]

ano_atual = datetime.now().year

anos_consulta = list(range(2020, ano_atual + 1))

max_paginas_por_ano = 20

# =========================================================
# STEP 4 — COLETA COM PAGINAÇÃO
# =========================================================

registros_contratos = []

for orgao in orgaos_consulta:

    codigo_orgao = orgao["codigo_orgao"]
    nome_orgao = orgao["nome_orgao"]

    print(f"\nConsultando órgão: {nome_orgao} ({codigo_orgao})")

    for ano in anos_consulta:

        data_inicial = f"01/01/{ano}"

        if ano == ano_atual:
            data_final = datetime.now().strftime("%d/%m/%Y")
        else:
            data_final = f"31/12/{ano}"

        print(f"\nPeríodo: {data_inicial} até {data_final}")

        pagina = 1

        while pagina <= max_paginas_por_ano:

            params_contratos = {
                "dataInicial": data_inicial,
                "dataFinal": data_final,
                "codigoOrgao": codigo_orgao,
                "pagina": pagina
            }

            resposta_contratos = requests.get(
                URL_CONTRATOS,
                headers=headers_transparencia,
                params=params_contratos
            )

            print(f"Página {pagina} - Status: {resposta_contratos.status_code}")

            if resposta_contratos.status_code != 200:
                print("Erro na consulta:")
                print(resposta_contratos.text[:1000])
                break

            dados_pagina = resposta_contratos.json()

            if not dados_pagina:
                print("Nenhum registro retornado. Encerrando este período.")
                break

            for registro in dados_pagina:
                registro["fonte_verificacao"] = "PORTAL_DA_TRANSPARENCIA"
                registro["tipo_dado"] = "CONTRATOS_PUBLICOS"
                registro["codigo_orgao_consultado"] = codigo_orgao
                registro["orgao_consultado"] = nome_orgao
                registro["periodo_inicial_consulta"] = data_inicial
                registro["periodo_final_consulta"] = data_final
                registro["pagina_coleta"] = pagina
                registro["url_consulta"] = resposta_contratos.url
                registro["data_coleta"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

            registros_contratos.extend(dados_pagina)

            print(f"Registros coletados nesta página: {len(dados_pagina)}")

            pagina += 1

            # Pequena pausa para não consultar rápido demais
            time.sleep(0.5)

# =========================================================
# STEP 5 — TRANSFORMAR EM DATAFRAME RAW
# =========================================================

df_transparencia_contratos_raw = pd.DataFrame(registros_contratos)

print(f"\nTotal de contratos coletados: {len(df_transparencia_contratos_raw)}")

df_transparencia_contratos_raw.head()

salvar_raw(df_transparencia_contratos_raw, "transparencia_contratos_mec")


Consultando órgão: MINISTERIO_DA_EDUCACAO (26000)

Período: 01/01/2020 até 31/12/2020
Página 1 - Status: 200
Registros coletados nesta página: 15
Página 2 - Status: 200
Registros coletados nesta página: 14
Página 3 - Status: 200
Nenhum registro retornado. Encerrando este período.

Período: 01/01/2021 até 31/12/2021
Página 1 - Status: 200
Registros coletados nesta página: 15
Página 2 - Status: 200
Registros coletados nesta página: 1
Página 3 - Status: 200
Nenhum registro retornado. Encerrando este período.

Período: 01/01/2022 até 31/12/2022
Página 1 - Status: 200
Registros coletados nesta página: 15
Página 2 - Status: 200
Registros coletados nesta página: 4
Página 3 - Status: 200
Nenhum registro retornado. Encerrando este período.

Período: 01/01/2023 até 31/12/2023
Página 1 - Status: 200
Registros coletados nesta página: 15
Página 2 - Status: 200
Registros coletados nesta página: 6
Página 3 - Status: 200
Nenhum registro retornado. Encerrando este período.

Período: 01/01/2024 até 31/

<h3>Despesas públicas</h3>

In [30]:
# =========================================================
# PORTAL DA TRANSPARÊNCIA - DESPESAS PÚBLICAS POR ÓRGÃO
# Objetivo: coletar despesas públicas por órgão superior e ano
# Fonte: Portal da Transparência
# Camada: RAW
# =========================================================

load_dotenv("../portal-transparencia-api-key.env")

API_KEY_TRANSPARENCIA = os.getenv("PORTAL_TRANSPARENCIA_API_KEY")

if not API_KEY_TRANSPARENCIA:
    raise ValueError("Chave do Portal da Transparência não encontrada.")

# =========================================================
# STEP 2 — CONFIGURAR ENDPOINT E HEADER
# O header 'chave-api-dados' é obrigatório na API
# =========================================================

URL_DESPESAS_ORGAO = "https://api.portaldatransparencia.gov.br/api-de-dados/despesas/por-orgao"

headers_transparencia = {
    "chave-api-dados": API_KEY_TRANSPARENCIA
}

# =========================================================
# STEP 3 — DEFINIR ÓRGÃOS E ANOS DA CONSULTA
# Usaremos os mesmos órgãos principais que você testou em contratos
# =========================================================

orgaos_consulta = [
    {
        "codigo_orgao": "26000",
        "nome_orgao": "MINISTERIO_DA_EDUCACAO"
    },
    {
        "codigo_orgao": "36000",
        "nome_orgao": "MINISTERIO_DA_SAUDE"
    },
    {
        "codigo_orgao": "30000",
        "nome_orgao": "MINISTERIO_DA_JUSTICA"
    },
    {
        "codigo_orgao": "55000",
        "nome_orgao": "MINISTERIO_DO_DESENVOLVIMENTO_SOCIAL"
    }
]

ano_atual = datetime.now().year

# Começa controlado. Depois podemos expandir para 2020 em diante.
anos_consulta = list(range(2023, ano_atual + 1))

max_paginas_por_ano = 20

# =========================================================
# STEP 4 — COLETA COM PAGINAÇÃO
# Para cada órgão e ano, buscamos página por página até:
# - acabar registro
# - atingir o limite definido em max_paginas_por_ano
# =========================================================

registros_despesas = []

for orgao in orgaos_consulta:

    codigo_orgao = orgao["codigo_orgao"]
    nome_orgao = orgao["nome_orgao"]

    print(f"\nConsultando órgão: {nome_orgao} ({codigo_orgao})")

    for ano in anos_consulta:

        print(f"\nAno: {ano}")

        pagina = 1

        while pagina <= max_paginas_por_ano:

            params_despesas = {
                "ano": ano,
                "orgaoSuperior": codigo_orgao,
                "pagina": pagina
            }

            resposta_despesas = requests.get(
                URL_DESPESAS_ORGAO,
                headers=headers_transparencia,
                params=params_despesas
            )

            print(f"Página {pagina} - Status: {resposta_despesas.status_code}")

            if resposta_despesas.status_code != 200:
                print("Erro na consulta:")
                print(resposta_despesas.text[:1000])
                break

            dados_pagina = resposta_despesas.json()

            if not dados_pagina:
                print("Nenhum registro retornado. Encerrando este órgão/ano.")
                break

            for registro in dados_pagina:
                registro["fonte_verificacao"] = "PORTAL_DA_TRANSPARENCIA"
                registro["tipo_dado"] = "DESPESAS_PUBLICAS_POR_ORGAO"
                registro["codigo_orgao_consultado"] = codigo_orgao
                registro["orgao_consultado"] = nome_orgao
                registro["ano_consulta"] = ano
                registro["pagina_coleta"] = pagina
                registro["url_consulta"] = resposta_despesas.url
                registro["data_coleta"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

            registros_despesas.extend(dados_pagina)

            print(f"Registros coletados nesta página: {len(dados_pagina)}")

            pagina += 1

            # Pausa para evitar excesso de requisições
            time.sleep(0.5)

# =========================================================
# STEP 5 — TRANSFORMAR EM DATAFRAME RAW
# =========================================================

df_transparencia_despesas_raw = pd.DataFrame(registros_despesas)

print(f"\nTotal de registros de despesas coletados: {len(df_transparencia_despesas_raw)}")

display(df_transparencia_despesas_raw.head())

salvar_raw(df_transparencia_despesas_raw, "transparencia_despesas_orgaos")


Consultando órgão: MINISTERIO_DA_EDUCACAO (26000)

Ano: 2023
Página 1 - Status: 200
Registros coletados nesta página: 15
Página 2 - Status: 200
Registros coletados nesta página: 15
Página 3 - Status: 200
Registros coletados nesta página: 15
Página 4 - Status: 200
Registros coletados nesta página: 15
Página 5 - Status: 200
Registros coletados nesta página: 15
Página 6 - Status: 200
Registros coletados nesta página: 15
Página 7 - Status: 200
Registros coletados nesta página: 15
Página 8 - Status: 200
Registros coletados nesta página: 12
Página 9 - Status: 200
Nenhum registro retornado. Encerrando este órgão/ano.

Ano: 2024
Página 1 - Status: 200
Registros coletados nesta página: 15
Página 2 - Status: 200
Registros coletados nesta página: 15
Página 3 - Status: 200
Registros coletados nesta página: 15
Página 4 - Status: 200
Registros coletados nesta página: 15
Página 5 - Status: 200
Registros coletados nesta página: 15
Página 6 - Status: 200
Registros coletados nesta página: 15
Página 7 -

,ano,orgao,codigoOrgao,orgaoSuperior,codigoOrgaoSuperior,empenhado,liquidado,pago,fonte_verificacao,tipo_dado,codigo_orgao_consultado,orgao_consultado,ano_consulta,pagina_coleta,url_consulta,data_coleta
0,2023,Fundo Nacional de Desenvolvimento da Educação,26298,Ministério da Educação,26000,"85.117.420.857,74","72.935.583.590,53","72.917.499.711,97",PORTAL_DA_TRANSPARENCIA,DESPESAS_PUBLICAS_POR_ORGAO,26000,MINISTERIO_DA_EDUCACAO,2023,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:26:30
1,2023,Empresa Brasileira de Serviços Hospitalares,26443,Ministério da Educação,26000,"10.416.077.643,66","9.535.227.966,02","8.837.850.471,77",PORTAL_DA_TRANSPARENCIA,DESPESAS_PUBLICAS_POR_ORGAO,26000,MINISTERIO_DA_EDUCACAO,2023,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:26:30
2,2023,Fundação Coordenação de Aperfeiçoamento de Pes...,26291,Ministério da Educação,26000,"5.265.239.235,43","4.729.488.875,63","4.721.140.325,36",PORTAL_DA_TRANSPARENCIA,DESPESAS_PUBLICAS_POR_ORGAO,26000,MINISTERIO_DA_EDUCACAO,2023,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:26:30
3,2023,Universidade Federal do Rio de Janeiro,26245,Ministério da Educação,26000,"4.403.037.484,33","4.324.107.853,24","3.952.657.059,33",PORTAL_DA_TRANSPARENCIA,DESPESAS_PUBLICAS_POR_ORGAO,26000,MINISTERIO_DA_EDUCACAO,2023,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:26:30
4,2023,Universidade Federal de Minas Gerais,26238,Ministério da Educação,26000,"2.779.986.882,31","2.701.001.911,59","2.448.998.943,35",PORTAL_DA_TRANSPARENCIA,DESPESAS_PUBLICAS_POR_ORGAO,26000,MINISTERIO_DA_EDUCACAO,2023,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:26:30



Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\transparencia_despesas_orgaos_raw_2026-05-13_00-26-55.csv
Total de registros extraídos: 536
Data e hora da extração: 13/05/2026 00:26:55


<h3>Emendas parlamentares</h3>

In [33]:
# =========================================================
# PORTAL DA TRANSPARÊNCIA - EMENDAS PARLAMENTARES
# Objetivo: coletar emendas parlamentares registradas no Portal da Transparência
# Fonte: Portal da Transparência
# Camada: RAW
# =========================================================

load_dotenv("../portal-transparencia-api-key.env")

API_KEY_TRANSPARENCIA = os.getenv("PORTAL_TRANSPARENCIA_API_KEY")

if not API_KEY_TRANSPARENCIA:
    raise ValueError("Chave do Portal da Transparência não encontrada.")

# =========================================================
# STEP 2 — CONFIGURAR ENDPOINT E HEADER
# Endpoint principal de emendas parlamentares
# =========================================================

URL_EMENDAS = "https://api.portaldatransparencia.gov.br/api-de-dados/emendas"

headers_transparencia = {
    "chave-api-dados": API_KEY_TRANSPARENCIA
}

# =========================================================
# STEP 3 — CONFIGURAR PAGINAÇÃO
# Começamos controlado para validar retorno e evitar requisições demais
# =========================================================

registros_emendas = []

pagina = 1
max_paginas = 100

# =========================================================
# STEP 4 — COLETA COM PAGINAÇÃO
# A API retorna uma lista por página.
# Quando vier lista vazia, encerramos a coleta.
# =========================================================

while pagina <= max_paginas:

    params_emendas = {
        "pagina": pagina
    }

    resposta_emendas = requests.get(
        URL_EMENDAS,
        headers=headers_transparencia,
        params=params_emendas
    )

    print(f"Página {pagina} - Status: {resposta_emendas.status_code}")

    if resposta_emendas.status_code != 200:
        print("Erro na consulta:")
        print(resposta_emendas.text[:1000])
        break

    dados_pagina = resposta_emendas.json()

    if not dados_pagina:
        print("Nenhum registro retornado. Encerrando paginação.")
        break

    for registro in dados_pagina:
        registro["fonte_verificacao"] = "PORTAL_DA_TRANSPARENCIA"
        registro["tipo_dado"] = "EMENDAS_PARLAMENTARES"
        registro["pagina_coleta"] = pagina
        registro["url_consulta"] = resposta_emendas.url
        registro["data_coleta"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

    registros_emendas.extend(dados_pagina)

    print(f"Registros coletados nesta página: {len(dados_pagina)}")

    pagina += 1

    # Pausa para respeitar o uso da API
    time.sleep(0.5)

# =========================================================
# STEP 5 — TRANSFORMAR EM DATAFRAME RAW
# =========================================================

df_transparencia_emendas_raw = pd.DataFrame(registros_emendas)

print(f"\nTotal de registros de emendas coletados: {len(df_transparencia_emendas_raw)}")

display(df_transparencia_emendas_raw.head())

salvar_raw(df_transparencia_emendas_raw, "transparencia_emendas_parlamentares")

Página 1 - Status: 200
Registros coletados nesta página: 15
Página 2 - Status: 200
Registros coletados nesta página: 15
Página 3 - Status: 200
Registros coletados nesta página: 15
Página 4 - Status: 200
Registros coletados nesta página: 15
Página 5 - Status: 200
Registros coletados nesta página: 15
Página 6 - Status: 200
Registros coletados nesta página: 15
Página 7 - Status: 200
Registros coletados nesta página: 15
Página 8 - Status: 200
Registros coletados nesta página: 15
Página 9 - Status: 200
Registros coletados nesta página: 15
Página 10 - Status: 200
Registros coletados nesta página: 15
Página 11 - Status: 200
Registros coletados nesta página: 15
Página 12 - Status: 200
Registros coletados nesta página: 15
Página 13 - Status: 200
Registros coletados nesta página: 15
Página 14 - Status: 200
Registros coletados nesta página: 15
Página 15 - Status: 200
Registros coletados nesta página: 15
Página 16 - Status: 200
Registros coletados nesta página: 15
Página 17 - Status: 200
Registros

,codigoEmenda,ano,tipoEmenda,autor,nomeAutor,numeroEmenda,localidadeDoGasto,funcao,subfuncao,valorEmpenhado,valorLiquidado,valorPago,valorRestoInscrito,valorRestoCancelado,valorRestoPago,fonte_verificacao,tipo_dado,pagina_coleta,url_consulta,data_coleta
0,201826760002,2018,Emenda Individual - Transferências com Finalid...,VINICIUS GURGEL,VINICIUS GURGEL,0002,AMAPÁ (UF),Segurança pública,Policiamento,"0,00","0,00","0,00","11.710,22","0,00","288.289,78",PORTAL_DA_TRANSPARENCIA,EMENDAS_PARLAMENTARES,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:32:49
1,202050170007,2020,Emenda de Comissão,COMISSAO DE FINANCAS E TRIBUTACAO - CFT,COMISSAO DE FINANCAS E TRIBUTACAO - CFT,0007,Nacional,Agricultura,Promoção da produção agropecuária,"0,00","0,00","0,00","11.385.738,68","1.569.992,02","3.195.992,49",PORTAL_DA_TRANSPARENCIA,EMENDAS_PARLAMENTARES,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:32:49
2,201524570012,2015,Emenda Individual - Transferências com Finalid...,RAUL HENRY,RAUL HENRY,0012,PERNAMBUCO (UF),Saúde,Assistência hospitalar e ambulatorial,"1,00","0,00","0,00","0,00","0,00","1,00",PORTAL_DA_TRANSPARENCIA,EMENDAS_PARLAMENTARES,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:32:49
3,202081001549,2020,Emenda de Relator,RELATOR GERAL,RELATOR GERAL,1549,Nacional,Agricultura,Promoção da produção agropecuária,"1,00","0,00","0,00","1,00","1,00","0,00",PORTAL_DA_TRANSPARENCIA,EMENDAS_PARLAMENTARES,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:32:49
4,201930600001,2019,Emenda Individual - Transferências com Finalid...,CABUCU BORGES,CABUCU BORGES,0001,AMAPÁ - AP,Saúde,Atenção básica,"175,00","0,00","0,00","175,00","0,00","175,00",PORTAL_DA_TRANSPARENCIA,EMENDAS_PARLAMENTARES,1,https://api.portaldatransparencia.gov.br/api-d...,13/05/2026 00:32:49



Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\transparencia_emendas_parlamentares_raw_2026-05-13_00-34-20.csv
Total de registros extraídos: 1500
Data e hora da extração: 13/05/2026 00:34:20


<h2>STF</h2>
<i> - rodar diariamente por ser consulta RSS

In [4]:
# =========================================================
# STF - NOTÍCIAS OFICIAIS
# Objetivo: coletar notícias oficiais do STF para o Pipeline 2
# Fonte: Notícias STF
# Camada: RAW
# =========================================================

import feedparser

# =========================================================
# STEP 1 — DEFINIR O FEED DO STF
# O portal de notícias do STF costuma usar estrutura de feed.
# Se não retornar registros, a gente investiga outro caminho.
# =========================================================

URL_STF_NOTICIAS = "https://noticias.stf.jus.br/feed/"

feed_stf = feedparser.parse(URL_STF_NOTICIAS)

print(f"Total de notícias encontradas no STF: {len(feed_stf.entries)}")

if feed_stf.bozo:
    print("Aviso: possível problema ao ler o feed do STF.")

# =========================================================
# STEP 2 — EXTRAIR OS DADOS DO FEED
# Aqui transformamos cada notícia em um registro tabular
# =========================================================

registros_stf = []

for noticia in feed_stf.entries:
    registros_stf.append({
        "fonte_verificacao": "STF",
        "tipo_dado": "NOTICIAS_OFICIAIS",
        "titulo": noticia.get("title", ""),
        "link": noticia.get("link", ""),
        "resumo": noticia.get("summary", ""),
        "data_publicacao": noticia.get("published", ""),
        "url_feed": URL_STF_NOTICIAS,
        "data_coleta": datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    })

df_stf_noticias_raw = pd.DataFrame(registros_stf)

print(f"Total de registros coletados: {len(df_stf_noticias_raw)}")

df_stf_noticias_raw.head()

salvar_raw(df_stf_noticias_raw, "stf_noticias_oficiais")

Total de notícias encontradas no STF: 10
Total de registros coletados: 10
Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\stf_noticias_oficiais_raw_2026-05-16_14-03-48.csv
Total de registros extraídos: 10
Data e hora da extração: 16/05/2026 14:03:48
